In [1]:
import pandas as pd
import numpy as np

# Batdongsan

In [3]:
bds_jan26 = pd.read_csv('output/batdongsan-7.11.2025-20.1.2026.csv')
bds_jul25 = pd.read_csv('output/batdongsan-07.2025.csv')
bds_oct2025 = pd.read_csv('output/batdongsan-10.2025.csv')

In [12]:
print(f'Jan 2026: {bds_jan26.columns}')
print(f'Jul 2025: {bds_jul25.columns}')
print(f'Oct 2025: {bds_oct2025.columns}')

Jan 2026: Index(['id', 'url', 'title', 'short_address', 'address_parts', 'latitude',
       'longitude', 'main_info', 'description', 'other_info', 'image_urls'],
      dtype='object')
Jul 2025: Index(['url', 'title', 'latitude', 'longitude', 'short_address',
       'address_parts', 'main_info', 'description', 'other_info',
       'image_urls'],
      dtype='object')
Oct 2025: Index(['url', 'title', 'id', 'latitude', 'longitude', 'short_address',
       'address_parts', 'main_info', 'description', 'other_info',
       'image_urls'],
      dtype='object')


In [11]:
bds_jan26.drop(columns=['scraping_time'], inplace=True)

In [20]:
# Triển khai ghép tháng 10/2025 vào tháng 11/2025-1/2026
bds = pd.concat([bds_jan26, bds_oct2025])
bds.to_csv('output/batdongsan_listing_details.csv', index=False, encoding='utf-8-sig')

# Onehousing

In [2]:
oh_jan26 = pd.read_csv('output/onehousing-20.1.2026.csv')
oh_dec25 = pd.read_excel('output/onehousing-30.12.2025-06.01.2026.xlsx')
oh_dec25.drop(columns='id', inplace=True)

In [3]:
print(f'Jan 26: {oh_jan26.columns}')
print(f'Dec 25: {oh_dec25.columns}')

Jan 26: Index(['property_id', 'property_url', 'listing_title', 'total_price',
       'unit_price', 'city', 'district', 'alley_width', 'features', 'latitude',
       'longitude', 'property_description', 'image_url'],
      dtype='object')
Dec 25: Index(['property_id', 'listing_title', 'total_price', 'unit_price',
       'property_url', 'image_url', 'city', 'district', 'alley_width',
       'features', 'property_description', 'has_updated', 'updated_at'],
      dtype='object')


Chốt lại là lấy theo schema cũ của Jan 2026

In [8]:
print(oh_jan26.shape[0])
print(oh_dec25.shape[0]) # bỏ cái này nè

969
1484


In [10]:
oh_jan26.to_csv('output/onehousing-listing-details.csv', index=False, encoding='utf-8-sig')

# Test duplicated rows

In [1]:
import sqlite3
import pandas as pd

with sqlite3.connect('output/real_estate.db') as conn:
    bds_raw = pd.read_sql('SELECT * FROM bds_raw', con = conn)
    oh_raw = pd.read_sql('SELECT * FROM onehousing_raw', con = conn)
    cleaned = pd.read_sql('SELECT * FROM cleaned', con = conn)

In [6]:
oh_raw.shape[0]

969

In [9]:
oh_raw.duplicated(subset=['property_id', 'listing_title', 'total_price', 'unit_price', 'city', 'district', 'alley_width', 'features', 'property_description']).sum()

np.int64(0)

In [55]:
oh_raw.duplicated().sum()

np.int64(0)

In [11]:
cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7340 entries, 0 to 7339
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Tỉnh/Thành phố                        7340 non-null   object 
 1   Thành phố/Quận/Huyện/Thị xã           7340 non-null   object 
 2   Xã/Phường/Thị trấn                    7340 non-null   object 
 3   Đường phố                             7340 non-null   object 
 4   Chi tiết                              7340 non-null   object 
 5   Nguồn thông tin                       7340 non-null   object 
 6   Tình trạng giao dịch                  7340 non-null   object 
 7   Thời điểm giao dịch/rao bán           7340 non-null   object 
 8   Thông tin liên hệ                     0 non-null      object 
 9   Giá rao bán/giao dịch                 7340 non-null   float64
 10  Giá ước tính                          7340 non-null   int64  
 11  Loại đơn giá (đ/m

# Test scrape onehousing

In [1]:
from Onehousing.orchestrator import (
    scrape_onehousing_urls as scrape_oh_urls, 
    scrape_onehousing_details as scrape_oh_details,
    process_onehousing_data
)
from commons.state_manager import PipelineStateManager, CircuitBreaker, PipelineStopException
from main import cleanup_intermediate_files

state_manager = PipelineStateManager()
circuit_breaker = CircuitBreaker() 

print("Starting New Pipeline Run...")
cleanup_intermediate_files() 
state_manager.reset_for_new_run()

print('Scrape URLs')
scrape_oh_urls(circuit_breaker=circuit_breaker, state_manager=state_manager)
print('Scrape details')
scrape_oh_details(circuit_breaker=circuit_breaker)

Starting New Pipeline Run...
Cleaning up old CSV files...
Scrape URLs
[Orchestrator] Starting 2 URL workers
[Writer] Saved batch. Total: 20
[Worker 1] Page 2: Found 20
[Writer] Saved batch. Total: 40
[Worker 0] Page 1: Found 20
[Writer] Saved 40 URLs
Scrape details
[Onehousing] Total URLs: 40, Done: 0, Pending: 40
[Orchestrator] Processing 40 URLs
[Details Writer] Creating new file: onehousing_listing_details.csv
[Writer] Saved batch. Total: 5
[Writer] Saved batch. Total: 10
[Writer] Saved batch. Total: 15
[Writer] Saved batch. Total: 20
[Writer] Saved batch. Total: 25
[Writer] Saved batch. Total: 30
[Writer] Saved batch. Total: 35
[Writer] Saved batch. Total: 40
[Writer] Saved 40 details


In [2]:
import pandas as pd

new_details = pd.read_csv('output/onehousing_listing_details.csv')
new_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   property_url          40 non-null     object 
 1   listing_title         40 non-null     object 
 2   property_id           40 non-null     object 
 3   total_price           40 non-null     object 
 4   unit_price            40 non-null     object 
 5   alley_width           31 non-null     object 
 6   image_url             40 non-null     object 
 7   city                  40 non-null     object 
 8   district              40 non-null     object 
 9   features              40 non-null     object 
 10  latitude              40 non-null     float64
 11  longitude             40 non-null     float64
 12  property_description  40 non-null     object 
dtypes: float64(2), object(11)
memory usage: 4.2+ KB
